# Retrieval checks

Exercises the `KnowledgeBase` retriever as a pipeline: parse constraints out of the question,
run a wide hybrid search with server-side filters, drop rows that violate a stated allergy
(checking `allergens_contains` **and** `allergens_may_contain`), rerank with Cohere, then
apply a relevance gate. No answer generation. Runs against the live collection.

Prerequisites: the collection is loaded (`scripts/load_knowledge_base.py`) and `.env` holds
the values `00_environment_check.ipynb` verifies. Reranking needs a Cohere key; a trial key
is rate-limited, so some cells may fall back to hybrid order.

## 1. Connect

Loads `.env`, connects to Weaviate Cloud with the vectorizer API-key header, and opens
`KnowledgeBase`. Prints the object count (197 for the full corpus). The client stays open
until the final cell.

In [ ]:
import json
import math
import os
import re
import time
import urllib.error
import urllib.request
from collections import Counter

import weaviate
from dotenv import find_dotenv, load_dotenv
from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, FilterReturn, MetadataQuery

load_dotenv(find_dotenv(usecwd=True))

PROVIDER = os.environ.get("EMBEDDING_PROVIDER", "cohere").lower()
COHERE_KEY = os.environ.get("EMBEDDING_API_KEY", "")
_hdr = "X-OpenAI-Api-Key" if PROVIDER == "openai" else "X-Cohere-Api-Key"

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={_hdr: COHERE_KEY},
)
kb = client.collections.get("KnowledgeBase")
print("connected -", kb.aggregate.over_all(total_count=True).total_count, "objects")

## 2. Pipeline

Tunables and the pipeline functions:

- `ALPHA = 0.75` — hybrid keyword/vector mix (Weaviate's own default); `K = 20` candidates;
  `TOP_N = 6` after rerank; `GATE = 0.15` minimum rerank score to count as answerable;
  `RERANK_MODEL = "rerank-v3.5"`.
- `parse_constraints(question)` — rule-based; returns a Weaviate filter for positive
  constraints (`vegan`, `under £N`) plus a set of allergens to exclude when the wording
  signals an allergy. Swap for an LLM call later without touching the rest.
- `retrieve()` — one wide hybrid query. `rerank()` — Cohere `rerank-v3.5`, with retry/back-off
  and a hybrid-order fallback on rate-limit. `search()` — parse → retrieve → allergen exclude
  → rerank → gate. `show()` / `show_search()` print hits with hybrid (`hy`) and rerank (`rr`)
  scores.

In [ ]:
ALPHA = 0.75
K = 20
TOP_N = 6
GATE = 0.15
RERANK_MODEL = "rerank-v3.5"

FIELDS = [
    "name",
    "category",
    "item_type",
    "price_gbp",
    "dietary_tags",
    "allergens_contains",
    "allergens_may_contain",
    "is_gluten_free_listed",
    "embedding_text",
]

NUT_ALLERGENS = {
    "peanuts",
    "tree nuts",
    "almond nuts",
    "walnuts",
    "hazelnuts",
    "pecan nuts",
    "pistachios",
    "brazil nuts",
    "cashew nuts",
    "macadamia nuts",
}
ALLERGEN_SYNONYMS: dict[str, set[str]] = {
    "peanut": {"peanuts"},
    "nut": NUT_ALLERGENS,
    "gluten": {"cereals containing gluten", "wheat", "barley", "oats", "rye"},
    "wheat": {"wheat", "cereals containing gluten"},
    "dairy": {"milk"},
    "milk": {"milk"},
    "egg": {"eggs"},
    "soy": {"soya"},
    "soya": {"soya"},
    "sesame": {"sesame"},
    "shellfish": {"crustaceans", "molluscs"},
    "crustacean": {"crustaceans"},
    "fish": {"fish"},
    "celery": {"celery"},
    "mustard": {"mustard"},
    "sulphite": {"sulphites"},
    "lupin": {"lupin"},
}
ALLERGY_CONTEXT = re.compile(r"allerg|intoleran|free|without|avoid|\bno\b")
PRICE_RE = re.compile(
    r"(?:under|below|less than|cheaper than|max|up to)\s*£?\s*(\d+(?:\.\d{1,2})?)"
)
INFO_RE = re.compile(r"\boptions?\b|do you (have|offer|serve|sell|take)|is there|allergen info")


def pstr(o, key: str) -> str:
    """properties[key] as a string, or "" if it is not one."""
    v = o.properties.get(key)
    return v if isinstance(v, str) else ""


def plist(o, key: str) -> list:
    """properties[key] as a list, or [] if it is not one."""
    v = o.properties.get(key)
    return v if isinstance(v, list) else []


def pnum(o, key: str) -> float | None:
    """properties[key] as a float, or None if it is not numeric."""
    v = o.properties.get(key)
    return float(v) if isinstance(v, (int, float)) else None


def allergen_set(o) -> set[str]:
    """Union of an object's allergens_contains and allergens_may_contain."""
    return {*plist(o, "allergens_contains"), *plist(o, "allergens_may_contain")}


def parse_constraints(question: str) -> tuple[FilterReturn | None, set[str], list[str]]:
    """Turn a question into a Weaviate filter, an allergen exclusion set, and notes."""
    ql = question.lower()
    clauses: list[FilterReturn] = []
    exclude: set[str] = set()
    notes: list[str] = []
    dish_intent = INFO_RE.search(ql) is None

    if dish_intent and re.search(r"\bvegan\b", ql):
        clauses.append(Filter.by_property("dietary_tags").contains_any(["vegan"]))
        notes.append("dietary=vegan")
    elif dish_intent and re.search(r"\b(vegetarian|veggie)\b", ql):
        clauses.append(Filter.by_property("dietary_tags").contains_any(["vegetarian"]))
        notes.append("dietary=vegetarian")

    price = PRICE_RE.search(ql)
    if dish_intent and price:
        cap = float(price.group(1))
        clauses.append(Filter.by_property("price_gbp").less_or_equal(cap))
        notes.append(f"price<=£{cap:g}")

    if ALLERGY_CONTEXT.search(ql):
        for term, values in ALLERGEN_SYNONYMS.items():
            if re.search(rf"\b{re.escape(term)}s?\b", ql):
                exclude |= values
                notes.append(f"exclude {term}")

    server = Filter.all_of(clauses) if clauses else None
    return server, exclude, notes


def retrieve(query: str, k: int = K, alpha: float = ALPHA, filters: FilterReturn | None = None):
    """Run one hybrid (keyword + vector) query against KnowledgeBase."""
    return kb.query.hybrid(
        query=query,
        alpha=alpha,
        limit=k,
        filters=filters,
        return_properties=FIELDS,
        return_metadata=MetadataQuery(score=True),
    ).objects


def rerank(query: str, objs: list, top_n: int = TOP_N) -> list[dict]:
    """Rerank objs against query with Cohere; fall back to hybrid order on rate limit."""
    if not objs:
        return []
    docs = [pstr(o, "embedding_text") or pstr(o, "name") for o in objs]
    payload = json.dumps(
        {"model": RERANK_MODEL, "query": query, "documents": docs, "top_n": min(top_n, len(docs))}
    ).encode()
    results = None
    for attempt in range(4):
        try:
            req = urllib.request.Request(
                "https://api.cohere.com/v2/rerank",
                data=payload,
                headers={
                    "Authorization": f"Bearer {COHERE_KEY.strip()}",
                    "Content-Type": "application/json",
                },
                method="POST",
            )
            with urllib.request.urlopen(req, timeout=30) as resp:
                results = json.load(resp)["results"]
            break
        except urllib.error.HTTPError as e:
            retryable = e.code == 429 and attempt < 3
            code = e.code
            e.close()
            if retryable:
                time.sleep(2 * 4**attempt)
            else:
                print(f"   [rerank HTTP {code}; using hybrid order]")
                break
    if results is None:
        return [
            {"obj": o, "rerank": math.nan, "hybrid": o.metadata.score or 0.0} for o in objs[:top_n]
        ]
    return [
        {
            "obj": objs[r["index"]],
            "rerank": float(r["relevance_score"]),
            "hybrid": objs[r["index"]].metadata.score or 0.0,
        }
        for r in results
    ]


def search(question: str, gate: float = GATE) -> dict:
    """Full pipeline: parse constraints, retrieve, exclude allergens, rerank, gate."""
    server, exclude, notes = parse_constraints(question)
    objs = retrieve(question, filters=server)
    kept = [o for o in objs if not (allergen_set(o) & exclude)] if exclude else objs
    ranked = rerank(question, kept)
    top = ranked[0]["rerank"] if ranked else 0.0
    answerable = bool(ranked) if math.isnan(top) else top >= gate
    return {
        "notes": notes,
        "excluded": sorted(exclude),
        "retrieved": len(objs),
        "kept": len(kept),
        "ranked": ranked,
        "top": top,
        "answerable": answerable,
    }


def line(o) -> str:
    """One-line summary of an object: type, name, category, price, dietary tags."""
    price = pnum(o, "price_gbp")
    price_s = f"  £{price:.2f}" if price is not None else ""
    diet = plist(o, "dietary_tags")
    diet_s = f"  {diet}" if diet else ""
    return f"[{pstr(o, 'item_type')}] {pstr(o, 'name')} <{pstr(o, 'category')}>{price_s}{diet_s}"


def show(label: str, objs: list) -> None:
    """Print raw hybrid hits under a label, with their hybrid score."""
    print(f"{label}  ({len(objs)} hits)")
    for o in objs:
        print(f"  hy={o.metadata.score or 0.0:.3f}  {line(o)}")
    print()


def show_search(question: str) -> None:
    """Run search() and print its constraints, verdict, and ranked hits."""
    r = search(question)
    if r["ranked"] and math.isnan(r["top"]):
        verdict = "RERANK UNAVAILABLE (hybrid order)"
    elif r["answerable"]:
        verdict = f"ANSWERABLE (top rr {r['top']:.3f})"
    else:
        verdict = f"NO CONFIDENT MATCH (top rr {r['top']:.3f} < {GATE})"
    print(f"q: {question!r}")
    print(f"   constraints {r['notes'] or '-'} | retrieved {r['retrieved']} -> kept {r['kept']}")
    print(f"   {verdict}")
    for h in r["ranked"]:
        print(f"   rr={h['rerank']:.3f}  hy={h['hybrid']:.3f}  {line(h['obj'])}")
    print()

## 3. Corpus vocabulary

Distinct `dietary_tags`, `allergens_contains`, and `allergens_may_contain` values with
counts, plus the category list. Nut allergens appear only in `allergens_may_contain` — no
dish declares one as an ingredient — which is why an allergy filter has to check both lists.

In [ ]:
tags: Counter = Counter()
contains: Counter = Counter()
may_contain: Counter = Counter()
cats: Counter = Counter()

_props = ["dietary_tags", "allergens_contains", "allergens_may_contain", "category"]
for o in kb.iterator(return_properties=_props):
    tags.update(plist(o, "dietary_tags"))
    contains.update(plist(o, "allergens_contains"))
    may_contain.update(plist(o, "allergens_may_contain"))
    cats[pstr(o, "category")] += 1

print("dietary_tags:", dict(tags))
print("allergens_contains:", dict(contains.most_common()))
print("allergens_may_contain:", dict(may_contain.most_common()))
print(f"categories ({len(cats)}):", ", ".join(sorted(cats)))

## 4. Natural-language menu queries

Guest-style menu requests through the full `search()` pipeline. Constraints in the wording
(`vegan`, `under £8`) are parsed into filters before retrieval.

In [ ]:
for q in [
    "something spicy with noodles",
    "vegan ramen",
    "a vegetarian main under £8",
    "light starter to share",
    "what cold desserts are there",
]:
    show_search(q)

## 5. FAQ queries

Questions that should resolve to `faq` rows, through the same pipeline.

In [ ]:
for q in [
    "what time do you open",
    "can I book a table for a large group",
    "where can I find allergen information",
    "do you sell gift cards",
]:
    show_search(q)

## 6. Keyword / vector balance

Raw hybrid retrieval, no rerank, at `alpha` 0 / 0.5 / 1 — the evidence for the `ALPHA = 0.75`
default. `alpha=0` (keyword only) is poor for paraphrased queries.

In [ ]:
q = "food that is not meat"
for a in (0.0, 0.5, 1.0):
    show(f"alpha={a}  {q!r}", retrieve(q, k=5, alpha=a))

## 7. Exact-name lookup

Raw hybrid at `alpha=0` — dish-name lookups, the case BM25 handles well.

In [ ]:
for q in ["yasai yaki soba", "chilli chicken ramen", "edamame", "katsu curry"]:
    show(repr(q), retrieve(q, k=3, alpha=0.0))

## 8. Constraint parsing and filters

### 8.1 Parsed constraints

`parse_constraints()` output for a few questions: whether a server-side filter was built, the
allergen exclusion set, and human-readable notes.

In [ ]:
for q in [
    "a vegan noodle dish under £10",
    "I have a nut allergy, what can I eat",
    "gluten free chicken",
    "something with prawns",
]:
    server, exclude, notes = parse_constraints(q)
    has_filter = "yes" if server is not None else "none"
    print(f"{q!r}")
    print(f"   filter={has_filter}  exclude={sorted(exclude) or '-'}  notes={notes}")

### 8.2 Dietary

`vegan` in the question adds a `dietary_tags` filter before retrieval.

In [ ]:
show_search("a vegan noodle dish")

### 8.3 Price

`under £N` in the question adds a `price_gbp <= N` filter; FAQ rows (null price) drop out.

In [ ]:
show_search("a filling main under £8")

### 8.4 Allergen-safe

A stated nut allergy removes every row listing any nut in `allergens_contains` **or**
`allergens_may_contain` — the exclusion a nut-allergic guest needs. The `may_contain` list of
each surviving row is printed to confirm it is clear.

In [ ]:
q = "a nut-free katsu curry"
r = search(q)
print(f"{q!r}")
print(f"   excluded: {r['excluded']}")
print(f"   retrieved {r['retrieved']} -> {r['kept']} nut-safe -> reranked {len(r['ranked'])}")
for h in r["ranked"]:
    o = h["obj"]
    print(f"   rr={h['rerank']:.3f}  {line(o)}")
    print(f"        may_contain={plist(o, 'allergens_may_contain')}")

## 9. Reranking effect

The same drift-prone query before and after Cohere rerank. Rerank scores are comparable
across queries; hybrid scores are not.

In [ ]:
q = "light starter to share"
show("raw hybrid  " + repr(q), retrieve(q, k=6))
print("after rerank:")
for h in rerank(q, retrieve(q)):
    print(f"   rr={h['rerank']:.3f}  hy={h['hybrid']:.3f}  {line(h['obj'])}")

## 10. Relevance gate

Out-of-domain and in-domain queries through `search()`. When the top rerank score is below
`GATE`, the pipeline reports NO CONFIDENT MATCH instead of handing hits to an answer step.

In [ ]:
for q in [
    "how do I change a car tyre",
    "what is the capital of France",
    "do you have vegan options",
]:
    show_search(q)

## 11. Variant-recipe hazard

Lists every `(... recipe)` row, then reranks the shared base name. The gluten-free / vegan
variants must stay as separate ranked hits, distinguishable by `allergens_contains`.

In [ ]:
_vp = ["name", "allergens_contains", "allergens_may_contain", "is_gluten_free_listed"]
variants = [o for o in kb.iterator(return_properties=_vp) if "recipe)" in pstr(o, "name")]

print(len(variants), "variant rows:")
for o in variants:
    print(f"  {pstr(o, 'name')}  contains={plist(o, 'allergens_contains')}")
print()

base = pstr(variants[0], "name").split(" (")[0] if variants else "signature seafood ramen"
print(f"rerank {base!r}:")
for h in rerank(base, retrieve(base)):
    print(f"   rr={h['rerank']:.3f}  {line(h['obj'])}")

## 12. Close connection

Releases the Weaviate client held open since cell 1.

In [ ]:
client.close()
print("closed")